# SCAPI Data API — Catalogs (product family)

Uses the **SCAPI Admin (Data) API** with **OAuth client-credentials** to read the
product family's top-level resource: **catalogs**.

> **Scope note.** In this SDK the SCAPI Admin APIs cover *catalog listing* and
> *sites* — there is **no** SCAPI Admin operation for product detail, product
> search, or content assets. For those, use the OCAPI Data API notebooks
> (`11-ocapi-products.ipynb`, `12-ocapi-content-assets.ipynb`).

Two ways are shown:
1. the high-level `create_catalogs_backend(..., preference="scapi")` helper, and
2. a low-level call to `product/catalogs/v1` parsed into the **generated Pydantic
   type** `b2c_tooling_sdk.clients.models.scapi_catalogs.Catalogs`.

Connection settings come from `../dw.json` (needs `shortCode`, `tenantId`,
`clientId`, `clientSecret`).

In [ ]:
from pathlib import Path

from b2c_tooling_sdk import ResolveConfigOptions, resolve_config

# --- Connection settings (read from dw.json) ---
DW_JSON = Path("../dw.json").resolve()

config = await resolve_config(options=ResolveConfigOptions(config_path=str(DW_JSON)))
instance = config.create_b2c_instance()

# SCAPI Admin needs shortCode + tenantId AND stateless OAuth (clientId/clientSecret).
scfg = instance.scapi_client_config
if scfg is None:
    print(
        "This instance is NOT SCAPI-capable.\n"
        "Add shortCode + tenantId to dw.json and use client-credentials OAuth "
        "(clientId + clientSecret). SCAPI Admin cannot use browser/user auth."
    )
else:
    print(f"SCAPI ready · short_code={scfg.short_code} · tenant={scfg.tenant_id}")

In [ ]:
# --- Parameters (edit these) ---
CATALOG_LIMIT = 50   # max catalogs to fetch/print

## 1. High-level helper (`create_catalogs_backend`)

`preference="scapi"` forces the SCAPI backend (it raises if the instance is not
SCAPI-capable). Use `"auto"` to prefer SCAPI with an OCAPI fallback. Returns a
list of `CatalogInfo` (an SDK convenience type).

In [ ]:
from b2c_tooling_sdk import CatalogsBackendConfig, ListCatalogsOptions, create_catalogs_backend

backend = create_catalogs_backend(CatalogsBackendConfig(instance=instance, preference="scapi"))
catalogs = await backend.list_catalogs(ListCatalogsOptions(count=CATALOG_LIMIT))

print(f"{len(catalogs)} catalog(s) via SCAPI:")
for c in catalogs:
    print(f"  {c.id:<28} {c.name or ''}{'  (online)' if c.online else ''}")

## 2. Low-level call + generated type

Build a typed SCAPI Catalogs client for `product/catalogs/v1` and call
`GET /organizations/{organizationId}/catalogs` directly, then parse the response
into the **generated** `Catalogs` model. The `x-b2c-scope-mode: read` header tells
the auth middleware to request read scopes (`sfcc.catalogs`).

In [ ]:
from b2c_tooling_sdk.clients.custom_apis import to_organization_id
from b2c_tooling_sdk.clients.middleware import SCOPE_MODE_HEADER
from b2c_tooling_sdk.clients.scapi_catalogs import (
    Catalogs,
    ScapiCatalogsClientConfig,
    create_scapi_catalogs_client,
)

# scfg (from instance.scapi_client_config) carries the scope-flexible auth strategy.
client = create_scapi_catalogs_client(
    ScapiCatalogsClientConfig(short_code=scfg.short_code, tenant_id=scfg.tenant_id),
    scfg.auth,
)
org_id = to_organization_id(scfg.tenant_id)

result = await client.get(
    "/organizations/{organizationId}/catalogs",
    {
        "params": {"path": {"organizationId": org_id}, "query": {"limit": CATALOG_LIMIT, "offset": 0}},
        "headers": {SCOPE_MODE_HEADER: "read"},
    },
)

if result.error or result.data is None:
    status = result.response.status_code if result.response is not None else "?"
    print(f"list catalogs: HTTP {status} — {str(result.error)[:200]}")
else:
    typed = Catalogs.model_validate(result.data)  # <- generated Pydantic model
    print(f"total={typed.total} limit={typed.limit} offset={typed.offset}")
    for c in typed.data or []:
        name = c.name.get("default") if isinstance(c.name, dict) else c.name
        print(f"  {c.id:<28} {name or ''}{'  (online)' if c.online else ''}")